# 01 Data Audit and Cleaning

This notebook audits the raw IBM Telco Customer Churn dataset, resolves the key data quality issues, and writes the cleaned modelling table to `data/processed/clean_telco_churn.csv`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
import pandas as pd

from src.preprocessing import clean_telco_data, save_clean_dataset

raw_path = DATA_DIR / 'raw' / 'telco_churn.csv'
df = pd.read_csv(raw_path)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print('Shape:', df.shape)
print('\nInfo:')
df.info()
print('\nDescribe:')
df.describe(include='all').transpose()

Shape: (7043, 21)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,3186-AJIEK,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
audit_df = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'dtype': df.dtypes.astype(str),
})
invalid_total_charges = pd.to_numeric(df['TotalCharges'], errors='coerce').isna().sum()
print('Duplicate rows:', int(df.duplicated().sum()))
print('Rows with invalid TotalCharges:', int(invalid_total_charges))
audit_df.sort_values('missing_values', ascending=False)

Duplicate rows: 0
Rows with invalid TotalCharges: 11


,missing_values,dtype
customerID,0,object
gender,0,object
SeniorCitizen,0,int64
Partner,0,object
Dependents,0,object
tenure,0,int64
PhoneService,0,object
MultipleLines,0,object
InternetService,0,object
OnlineSecurity,0,object


## Cleaning Decisions

- `TotalCharges` is stored as text and contains blank values, so it is converted to numeric with coercion.
- Rows with invalid `TotalCharges` are removed because the field is an important commercial predictor.
- `customerID` is dropped to prevent identifier leakage.
- Duplicate rows are removed before modelling.

In [4]:
clean_df, audit_summary = clean_telco_data(df)
pd.DataFrame([audit_summary])

,initial_rows,duplicate_rows_removed,invalid_total_charges_removed,rows_after_cleaning
0,7043,0,11,7032


In [5]:
print('Post-cleaning shape:', clean_df.shape)
print('Remaining duplicates:', int(clean_df.duplicated().sum()))
display(clean_df.isna().sum())
clean_df.head()

Post-cleaning shape: (7032, 20)
Remaining duplicates: 22


gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
clean_path = DATA_DIR / 'processed' / 'clean_telco_churn.csv'
save_clean_dataset(clean_path)
print(f'Clean dataset saved to: {clean_path}')

Clean dataset saved to: E:\Academic Year\Projects\Customer Churn\customer-churn-prediction-system\data\processed\clean_telco_churn.csv


## Commentary

The cleaned dataset preserves the revenue, tenure, contract, and service signals required for churn modelling while removing malformed billing values and identifier noise. This gives the later EDA and ML stages a reliable base table.